In [ ]:
# pip install torch==2.4.1 torchaudio==2.4.1 soundfile matplotlib numpy

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
from shutil import copy2
import matplotlib.pyplot as plt

In [ ]:
# Modelo Unet 2D para Audio Super Resolution

class AttentionGate(nn.Module):
    """Módulo de atención para ponderar skip connections"""
    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        # Basado en el paper "https://arxiv.org/abs/1804.03999"
        # https://github.com/ozan-oktay/Attention-Gated-Networks
        self.W_g = nn.Conv2d(F_g, F_int, kernel_size=1)
        self.W_x = nn.Conv2d(F_l, F_int, kernel_size=1)
        self.psi = nn.Conv2d(F_int, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        # Interpolar g para que tenga el mismo tamaño que x
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        # Calcular atención
        att = self.sigmoid(self.psi(self.relu(self.W_g(g) + self.W_x(x))))

        return x * att


class DilatedBlock(nn.Module):
    """Bloque con capas convolucionales dilatadas para capturar contexto de largo alcance sin perder resolución."""
    def __init__(self, channels):

        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(3,1), dilation=(1,1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(6,2), dilation=(2,2)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=(7,3), padding=(12,4), dilation=(4,4)),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x) + x


class ResBlock(nn.Module):
    """Bloque Residual para Attention ResUNet."""
    def __init__(self, in_ch, out_ch):
        
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm1 = nn.GroupNorm(out_ch//4, out_ch) # GroupNorm para batchs pequeños
        self.relu1 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=(7,3), padding=(3,1))
        self.norm2 = nn.GroupNorm(out_ch//4, out_ch)
        self.relu2 = nn.LeakyReLU(0.2, inplace=True)
        
        # Skip connection
        if in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.GroupNorm(out_ch//4, out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu1(out)
        
        out = self.conv2(out)
        out = self.norm2(out)
        
        out += identity
        out = self.relu2(out)
        
        return out


class UNetAudio2D(nn.Module):
    """Arquitectura Attention Res-UNet 2D adaptada para audio super resolution."""
    def __init__(self):

        super().__init__()

        # Encoder
        # Entrada: (B, 2, F, T)
        self.enc1 = ResBlock(2, 32)
        self.pool1 = nn.MaxPool2d((2,2))

        self.enc2 = ResBlock(32, 64)
        self.pool2 = nn.MaxPool2d((2,2))

        self.enc3 = ResBlock(64, 128)
        self.pool3 = nn.MaxPool2d((2,2))

        self.enc4 = ResBlock(128, 256)
        self.pool4 = nn.MaxPool2d((2,2))

        # Bottleneck
        self.bottleneck_conv = ResBlock(256, 512)
        self.bottleneck_dilated = DilatedBlock(512)

        # Decoder
        self.up4 = self.up_block(512,256)
        self.dec4 = ResBlock(512,256)

        self.up3 = self.up_block(256,128)
        self.dec3 = ResBlock(256,128)

        self.up2 = self.up_block(128,64)
        self.dec2 = ResBlock(128,64)

        self.up1 = self.up_block(64,32)
        self.dec1 = ResBlock(64,32)

        # Attention gates
        self.att4 = AttentionGate(256,256,128)
        self.att3 = AttentionGate(128,128,64)
        self.att2 = AttentionGate(64,64,32)
        self.att1 = AttentionGate(32,32,16)

        # Output
        self.final = nn.Conv2d(32,2,kernel_size=1)

    def up_block(self, in_ch, out_ch):
        """Bloque de upsampling en frecuencia y tiempo"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),  # sub-pixel upsampling
            nn.LeakyReLU(0.2, inplace=True)
        )

    def match_size(self, x, ref):
        """Asegura que x tenga el mismo tamaño que ref"""
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

        return x

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck_conv(self.pool4(e4))
        b = self.bottleneck_dilated(b)

        # Decoder con skip connections y attention gates
        up4 = self.match_size(self.up4(b), e4)
        d4 = self.dec4(torch.cat([up4, self.att4(up4, e4)], dim=1))

        up3 = self.match_size(self.up3(d4), e3)
        d3 = self.dec3(torch.cat([up3, self.att3(up3, e3)], dim=1))

        up2 = self.match_size(self.up2(d3), e2)
        d2 = self.dec2(torch.cat([up2, self.att2(up2, e2)], dim=1))

        up1 = self.match_size(self.up1(d2), e1)
        d1 = self.dec1(torch.cat([up1, self.att1(up1, e1)], dim=1))

        return self.final(d1) + x

In [ ]:
# Script para realizar inferencia
# Este script cargará el modelo entrenado y generará resultados

MODEL_PATH = 'unet2D_superres.pt'   # Archivo del modelo entrenado
INF_DIR = './data/inference'        # Archivos de entrada
OUTPUT_DIR = './results'            # Archivos de salida

TARGET_SR = 44100                   # Target sample rate
POOL_FACTOR = 16                    # 2^4 para 4 capas de pooling en UNet 2D
N_FFT = 1024                        # Tamaño de la FFT para STFT
HOP_LENGTH = 256                    # Salto entre ventanas STFT
FRAGMENT_LENGTH = 65536             # Longitud de los fragmentos en muestras

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def save_audio(tensor, path, sample_rate):
    """Guarda una forma de onda en un archivo de audio."""
    if tensor.ndim == 3:
        tensor = tensor.squeeze(0)
    elif tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)
    torchaudio.save(path, tensor.cpu(), sample_rate, bits_per_sample=16, encoding="PCM_S")


def waveform_to_stft(waveform, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Convierte una forma de onda a un STFT con canales real e imaginario."""
    if waveform.ndim == 2:
        waveform = waveform.squeeze(0)

    window = torch.hann_window(n_fft, device=DEVICE)
    stft = torch.stft(
        waveform,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        window=window,
        return_complex=True,
    )
    # Devolver como tensor de 2 canales: real e imaginario
    return torch.stack([stft.real, stft.imag], dim=0).to(original_device)

def stft_to_waveform(stft, n_fft=N_FFT, hop_length=HOP_LENGTH, length=None):
    """Convierte un STFT con canales real e imaginario de vuelta a forma de onda usando ISTFT."""

    stft_complex = torch.complex(stft[0], stft[1])
    window = torch.hann_window(n_fft, device=DEVICE)
    waveform = torch.istft(
        stft_complex,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        window=window,
        length=length,  # Forzar longitud exacta para evitar desfases en el corte
    )
    return waveform.unsqueeze(0)  # (1, T)


def normalize_stft(stft):
    """Log-compresión solo de la magnitud, preservando la fase."""
    real, imag = stft[0], stft[1]
    magnitude = torch.sqrt(real**2 + imag**2 + 1e-8)
    phase_cos = real / magnitude
    phase_sin = imag / magnitude
    mag_compressed = torch.log1p(magnitude)
    return torch.stack([mag_compressed * phase_cos, mag_compressed * phase_sin], dim=0)

def denormalize_stft(stft):
    """Inversa de log-compresión."""
    real, imag = stft[0], stft[1]
    mag_compressed = torch.sqrt(real**2 + imag**2 + 1e-8)
    phase_cos = real / mag_compressed
    phase_sin = imag / mag_compressed
    magnitude = torch.exp(mag_compressed) - 1
    return torch.stack([magnitude * phase_cos, magnitude * phase_sin], dim=0)


def pad_stft(stft, pool_factor=POOL_FACTOR):
    """Ajusta el STFT para que sea divisible por el factor de pooling."""
    _, freq_bins, time_frames = stft.shape
    pad_f = (pool_factor - (freq_bins % pool_factor)) % pool_factor
    pad_t = (pool_factor - (time_frames % pool_factor)) % pool_factor

    if pad_f > 0 or pad_t > 0:
        stft = torch.nn.functional.pad(stft, (0, pad_t, 0, pad_f), mode='reflect')

    return stft, freq_bins, time_frames


def save_waveform_plot(lr, sr, filename, sample_rate):
    """Genera un gráfico de la forma de onda de entrada y salida."""
    wave_lr = lr.squeeze().cpu().numpy()
    wave_sr = sr.squeeze().cpu().numpy()

    time_lr = np.linspace(0, len(wave_lr) / sample_rate, len(wave_lr))
    time_sr = np.linspace(0, len(wave_sr) / sample_rate, len(wave_sr))

    fig, axs = plt.subplots(2, 1, figsize=(12, 6))

    axs[0].plot(time_lr, wave_lr, color='steelblue', linewidth=0.5)
    axs[0].set_title("Input (Low Res)")
    axs[0].set_ylabel("Amplitud")
    axs[0].set_ylim(-1, 1)
    axs[0].grid(True, alpha=0.3)

    axs[1].plot(time_sr, wave_sr, color='darkorange', linewidth=0.5)
    axs[1].set_title("Output (Super Res)")
    axs[1].set_ylabel("Amplitud")
    axs[1].set_xlabel("Tiempo (s)")
    axs[1].set_ylim(-1, 1)
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


def save_spectrogram_plot(lr_waveform, sr_waveform, filename, sample_rate, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Genera un gráfico del espectrograma de entrada y salida."""
    spec_transform = torchaudio.transforms.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2)

    spec_lr = spec_transform(lr_waveform.cpu()).squeeze().numpy()
    spec_sr = spec_transform(sr_waveform.cpu()).squeeze().numpy()

    # Escala dB
    spec_lr_db = 10 * np.log10(spec_lr + 1e-10)
    spec_sr_db = 10 * np.log10(spec_sr + 1e-10)

    vmin = min(spec_lr_db.min(), spec_sr_db.min())
    vmax = max(spec_lr_db.max(), spec_sr_db.max())

    nyquist = sample_rate / 2
    duration_lr = (spec_lr.shape[1] * hop_length) / sample_rate
    duration_sr = (spec_sr.shape[1] * hop_length) / sample_rate
    extent_lr = [0, duration_lr, 0, nyquist]
    extent_sr = [0, duration_sr, 0, nyquist]

    fig, axs = plt.subplots(2, 1, figsize=(10, 8))

    im1 = axs[0].imshow(spec_lr_db, origin='lower', aspect='auto', cmap='magma', extent=extent_lr, vmin=vmin, vmax=vmax)
    axs[0].set_title("Input (Low Res)")
    axs[0].set_ylabel("Frecuencia (Hz)")
    axs[0].set_ylim(0, nyquist)
    fig.colorbar(im1, ax=axs[0], label="Amplitud (dB)")

    im2 = axs[1].imshow(spec_sr_db, origin='lower', aspect='auto', cmap='magma', extent=extent_sr, vmin=vmin, vmax=vmax)
    axs[1].set_title("Output (Super Res)")
    axs[1].set_ylabel("Frecuencia (Hz)")
    axs[1].set_ylim(0, nyquist)
    axs[1].set_xlabel("Tiempo (s)")
    fig.colorbar(im2, ax=axs[1], label="Amplitud (dB)")

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


def process_audio_in_chunks(model, stft, orig_f, orig_t, chunk_frames, overlap=32):
    """
    Procesa el STFT por chunks con overlap para evitar artefactos en los bordes.
    chunk_frames debe ser compatible con pool_factor=16.
    """
    _, F, T = stft.shape
    hop = chunk_frames - overlap
    output = torch.zeros_like(stft)
    weights = torch.zeros(T, device=stft.device)

    # Ventana trapezoidal
    window = torch.ones(chunk_frames, device=stft.device)
    if overlap > 0:
        window[:overlap] = torch.linspace(0, 1, overlap, device=stft.device)
        window[-overlap:] = torch.linspace(1, 0, overlap, device=stft.device)

    start = 0
    while start < T:
        end = min(start + chunk_frames, T)
        chunk = stft[:, :, start:end]

        # Pad si el chunk es más corto que chunk_frames
        pad_t = chunk_frames - chunk.shape[-1]
        if pad_t > 0:
            chunk = torch.nn.functional.pad(chunk, (0, pad_t))

        with torch.no_grad():
            pred_chunk = model(chunk.unsqueeze(0)).squeeze(0)

        actual_len = end - start
        
        w = window[:actual_len].clone()
        # Evitar fade-in en el primer bloque de audio
        if start == 0 and overlap > 0:
             w[:overlap] = 1.0
        # Evitar fade-out en el último bloque de audio
        if end == T and overlap > 0:
             w[-overlap:] = 1.0

        output[:, :, start:end] += pred_chunk[:, :, :actual_len] * w
        weights[start:end] += w

        start += hop

    # Normalizar por los pesos acumulados
    output = output / weights.unsqueeze(0).unsqueeze(0)
    return output


def inference():
    """Realiza inferencia en los archivos de entrada y guardar los resultados."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Cargar modelo
    print(f"Cargando modelo desde {MODEL_PATH}...")
    model = UNetAudio2D().to(DEVICE)
    try:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)) # weights_only=False para compatibilidad con AMD
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo del modelo '{MODEL_PATH}'.")
        return
    model.eval()

    # Procesar archivos de inferencia
    files = [f for f in os.listdir(INF_DIR) if f.endswith('.wav')]
    if not files:
        print(f"No se encontraron archivos .wav en {INF_DIR}")
        return

    print(f"Encontrados {len(files)} archivos para procesar.")

    resamplers = {}

    for filename in files:
        file_path = os.path.join(INF_DIR, filename)
        print(f"Procesando: {filename}")

        waveform, original_sr = torchaudio.load(file_path)

        # Resampleo a TARGET_SR si es necesario
        if original_sr != TARGET_SR:
            if original_sr not in resamplers:
                resamplers[original_sr] = torchaudio.transforms.Resample(original_sr, TARGET_SR)
            waveform = resamplers[original_sr](waveform)

        # Pasar a mono
        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Normalización por amplitud máxima
        max_val = waveform.abs().max().item() + 1e-8
        waveform_norm = waveform / max_val

        original_length = waveform_norm.size(1)
        input_for_plot = waveform_norm.clone()

        # Preparación STFT
        stft_input = waveform_to_stft(waveform_norm.to(DEVICE))
        stft_input = normalize_stft(stft_input)  # Log-compresión (igual que en dataset)
        stft_padded, orig_f, orig_t = pad_stft(stft_input)

        chunk_frames = FRAGMENT_LENGTH // HOP_LENGTH  # frames equivalentes a FRAGMENT_LENGTH muestras

        predicted_stft = process_audio_in_chunks(model, stft_padded, orig_f, orig_t, chunk_frames=chunk_frames)
        predicted_stft = predicted_stft[:, :orig_f, :orig_t]

        predicted_stft = denormalize_stft(predicted_stft)  # Inversa de log-compresión

        # ISTFT con longitud exacta para evitar artefactos de corte
        predicted_waveform = stft_to_waveform(predicted_stft, length=original_length)  # (1, T)

        # Prevención de valores NaN o infinitos que puedan surgir por la red o la ISTFT
        predicted_waveform = torch.nan_to_num(predicted_waveform, nan=0.0, posinf=1.0, neginf=-1.0)
        predicted_waveform = torch.clamp(predicted_waveform, -1.0, 1.0)

        base_name = os.path.splitext(filename)[0]
        save_path = os.path.join(OUTPUT_DIR, base_name)
        os.makedirs(save_path, exist_ok=True)

        # Guardar resultados
        copy2(file_path, os.path.join(save_path, 'input.wav'))
        save_audio(predicted_waveform, os.path.join(save_path, 'super_res.wav'), TARGET_SR)

        # Guardar gráficos de forma de onda y espectrograma
        save_waveform_plot(
            input_for_plot,
            predicted_waveform,
            os.path.join(save_path, 'waveform.png'),
            TARGET_SR,
        )

        save_spectrogram_plot(
            input_for_plot,
            predicted_waveform,
            os.path.join(save_path, 'spectrogram.png'),
            sample_rate=TARGET_SR,
        )

        print(f"Guardado en {save_path.replace(os.sep, '/')}/")

    print(f"Listo. Revisa la carpeta '{OUTPUT_DIR.replace(os.sep, '/')}'.")

In [ ]:
try:
    inference()
except KeyboardInterrupt:
    exit()
except Exception as e:
    DEVICE = 'cpu'
    print(f"Error durante la inferencia. Intentando en CPU...\n{e}")
    inference()